In [16]:
# Cell 1 — pick folder (Windows)
from pathlib import Path
import re, json
from datetime import datetime
import tkinter as tk
from tkinter import filedialog

def pick_folder(title="Select folder containing CSC*.ncs files"):
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    folder = filedialog.askdirectory(title=title)
    root.destroy()
    if not folder:
        raise RuntimeError("No folder selected.")
    return Path(folder)

folder = pick_folder()
folder

WindowsPath('D:/Data/Chronic Recordings m5/Rearranged Channles/m5s42jan25/2024-01-25_16-20-33')

In [17]:
# Cell 2 — paste mapping here
# mapping[i-1] = new channel number for old CSCi
mapping = [
    106,102,111,112,123,118,110,125,85,80,70,87,78,95,86,66,
    117,101,103,124,126,116,107,91,88,94,77,76,84,71,92,93,
    114,104,109,121,108,113,100,119,128,83,82,67,73,65,96,69,
    97,105,99,115,120,98,127,122,79,74,90,81,75,89,68,72,
    62,47,48,43,54,46,61,33,6,21,24,14,31,52,30,13,
    64,53,37,39,60,42,59,16,23,15,22,20,2,12,7,19,
    38,45,50,40,57,44,49,36,55,28,29,18,3,32,9,1,
    41,35,51,56,34,63,58,5,10,26,17,11,25,4,8,27
]

# Validate it's a true permutation of 1..128
assert len(mapping) == 128
assert set(mapping) == set(range(1,129)), "Mapping must be a permutation of 1..128"
print("Mapping validated: permutation of 1..128 ✅")

Mapping validated: permutation of 1..128 ✅


In [18]:
# Cell 3 — scan & build plan
pattern = re.compile(r"^CSC(\d+)\.ncs$", re.IGNORECASE)

def find_csc_files(folder: Path):
    files = []
    for p in folder.iterdir():
        if p.is_file():
            m = pattern.match(p.name)
            if m:
                ch = int(m.group(1))
                if 1 <= ch <= 128:
                    files.append((ch, p))
    return sorted(files, key=lambda x: x[0])

found = find_csc_files(folder)
print(f"Found {len(found)} CSC*.ncs files in 1..128 range.")

rename_plan = {}
for old_ch, old_path in found:
    new_ch = mapping[old_ch - 1]
    final_path = old_path.with_name(f"CSC{new_ch}.ncs")
    rename_plan[old_path] = final_path

list(rename_plan.items())[:5]

Found 128 CSC*.ncs files in 1..128 range.


[(WindowsPath('D:/Data/Chronic Recordings m5/Rearranged Channles/m5s42jan25/2024-01-25_16-20-33/CSC1.ncs'),
  WindowsPath('D:/Data/Chronic Recordings m5/Rearranged Channles/m5s42jan25/2024-01-25_16-20-33/CSC106.ncs')),
 (WindowsPath('D:/Data/Chronic Recordings m5/Rearranged Channles/m5s42jan25/2024-01-25_16-20-33/CSC2.ncs'),
  WindowsPath('D:/Data/Chronic Recordings m5/Rearranged Channles/m5s42jan25/2024-01-25_16-20-33/CSC102.ncs')),
 (WindowsPath('D:/Data/Chronic Recordings m5/Rearranged Channles/m5s42jan25/2024-01-25_16-20-33/CSC3.ncs'),
  WindowsPath('D:/Data/Chronic Recordings m5/Rearranged Channles/m5s42jan25/2024-01-25_16-20-33/CSC111.ncs')),
 (WindowsPath('D:/Data/Chronic Recordings m5/Rearranged Channles/m5s42jan25/2024-01-25_16-20-33/CSC4.ncs'),
  WindowsPath('D:/Data/Chronic Recordings m5/Rearranged Channles/m5s42jan25/2024-01-25_16-20-33/CSC112.ncs')),
 (WindowsPath('D:/Data/Chronic Recordings m5/Rearranged Channles/m5s42jan25/2024-01-25_16-20-33/CSC5.ncs'),
  WindowsPath('D

In [19]:
# Cell 4 — SAFE 2-step rename (prevents collisions)
def safe_rename_files(folder: Path, rename_plan: dict[Path, Path], dry_run: bool = True):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = folder / f"rename_log_{ts}.json"

    # Step A: old -> temp unique names
    temp_plan = {}
    for old_path in rename_plan:
        tmp = old_path.with_name(f"__TMP__{ts}__{old_path.name}")
        if tmp.exists():
            raise FileExistsError(f"Temp already exists: {tmp.name}")
        temp_plan[old_path] = tmp

    # Check for any external conflicts (target exists but isn't one of the sources)
    sources = set(rename_plan.keys())
    for final_path in set(rename_plan.values()):
        if final_path.exists() and final_path not in sources:
            raise FileExistsError(f"Target already exists and is not being moved away: {final_path.name}")

    # Print plan
    print("Step A (old → temp):")
    for old_path, tmp_path in sorted(temp_plan.items(), key=lambda x: x[0].name.lower()):
        print(f"  {old_path.name}  ->  {tmp_path.name}")

    print("\nStep B (temp → final):")
    for old_path, tmp_path in sorted(temp_plan.items(), key=lambda x: x[0].name.lower()):
        print(f"  {tmp_path.name}  ->  {rename_plan[old_path].name}")

    if dry_run:
        print(f"\nDRY RUN: nothing renamed.\nLog would be written to: {log_path.name}")
        return

    # Execute A
    for old_path, tmp_path in temp_plan.items():
        old_path.rename(tmp_path)

    # Execute B
    for old_path, tmp_path in temp_plan.items():
        tmp_path.rename(rename_plan[old_path])

    # Log
    log = {
        "folder": str(folder),
        "timestamp": ts,
        "stepA_old_to_temp": {str(k): str(v) for k, v in temp_plan.items()},
        "stepB_temp_to_final": {str(v): str(rename_plan[k]) for k, v in temp_plan.items()},
    }
    with open(log_path, "w", encoding="utf-8") as f:
        json.dump(log, f, indent=2)

    print(f"\nDONE ✅ Log saved to: {log_path}")

# Always do a dry-run first:
safe_rename_files(folder, rename_plan, dry_run=True)

Step A (old → temp):
  CSC1.ncs  ->  __TMP__20260223_174654__CSC1.ncs
  CSC10.ncs  ->  __TMP__20260223_174654__CSC10.ncs
  CSC100.ncs  ->  __TMP__20260223_174654__CSC100.ncs
  CSC101.ncs  ->  __TMP__20260223_174654__CSC101.ncs
  CSC102.ncs  ->  __TMP__20260223_174654__CSC102.ncs
  CSC103.ncs  ->  __TMP__20260223_174654__CSC103.ncs
  CSC104.ncs  ->  __TMP__20260223_174654__CSC104.ncs
  CSC105.ncs  ->  __TMP__20260223_174654__CSC105.ncs
  CSC106.ncs  ->  __TMP__20260223_174654__CSC106.ncs
  CSC107.ncs  ->  __TMP__20260223_174654__CSC107.ncs
  CSC108.ncs  ->  __TMP__20260223_174654__CSC108.ncs
  CSC109.ncs  ->  __TMP__20260223_174654__CSC109.ncs
  CSC11.ncs  ->  __TMP__20260223_174654__CSC11.ncs
  CSC110.ncs  ->  __TMP__20260223_174654__CSC110.ncs
  CSC111.ncs  ->  __TMP__20260223_174654__CSC111.ncs
  CSC112.ncs  ->  __TMP__20260223_174654__CSC112.ncs
  CSC113.ncs  ->  __TMP__20260223_174654__CSC113.ncs
  CSC114.ncs  ->  __TMP__20260223_174654__CSC114.ncs
  CSC115.ncs  ->  __TMP__20260223

In [20]:
# Cell 5 — run for real (ONLY if dry-run looks correct)
safe_rename_files(folder, rename_plan, dry_run=False)

Step A (old → temp):
  CSC1.ncs  ->  __TMP__20260223_174709__CSC1.ncs
  CSC10.ncs  ->  __TMP__20260223_174709__CSC10.ncs
  CSC100.ncs  ->  __TMP__20260223_174709__CSC100.ncs
  CSC101.ncs  ->  __TMP__20260223_174709__CSC101.ncs
  CSC102.ncs  ->  __TMP__20260223_174709__CSC102.ncs
  CSC103.ncs  ->  __TMP__20260223_174709__CSC103.ncs
  CSC104.ncs  ->  __TMP__20260223_174709__CSC104.ncs
  CSC105.ncs  ->  __TMP__20260223_174709__CSC105.ncs
  CSC106.ncs  ->  __TMP__20260223_174709__CSC106.ncs
  CSC107.ncs  ->  __TMP__20260223_174709__CSC107.ncs
  CSC108.ncs  ->  __TMP__20260223_174709__CSC108.ncs
  CSC109.ncs  ->  __TMP__20260223_174709__CSC109.ncs
  CSC11.ncs  ->  __TMP__20260223_174709__CSC11.ncs
  CSC110.ncs  ->  __TMP__20260223_174709__CSC110.ncs
  CSC111.ncs  ->  __TMP__20260223_174709__CSC111.ncs
  CSC112.ncs  ->  __TMP__20260223_174709__CSC112.ncs
  CSC113.ncs  ->  __TMP__20260223_174709__CSC113.ncs
  CSC114.ncs  ->  __TMP__20260223_174709__CSC114.ncs
  CSC115.ncs  ->  __TMP__20260223